# 🖥️ GRU -(CIFAR-10 + Selective Copying)

**This notebook runs CIFAR-10 ve Selective Copying set up designed equally to compare Mamba.**



### CIFAR-10 Config (20 epochs)
| Comb | d_model | n_layers | lr | Mamba Acc|
|------|---------|----------|-------|----------|
| 1 | 64 | 4 | 0.0005 | 0.46 |
| 2 | 128 | 2 | 0.0005 | 0.43 |

### Selective Copying Config (80 epochs)
| Comb | d_model | n_layers | lr | Mamba Acc |
|------|---------|----------|-------|----------|
| 1 | 64 | 4 | 0.001 | 0.3385 |
| 2 | 128 | 2 | 0.0005 | 0.3529 |

In [1]:
import time
notebook_start_time = time.time()

In [2]:
# HPC'de module load veya conda activate gerekebilir
# !module load cuda/11.8
# !conda activate mamba_env

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import lightning as L
from huggingface_hub import hf_hub_download
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.5.1
CUDA: True
GPU: Tesla V100-SXM2-32GB


In [3]:
NUM_WORKERS = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CIFAR-10 Config - FROM PRESENTATION
CIFAR10_CONFIG = {
    "combination_1": {"d_model": 64,  "n_layers": 4, "lr": 5e-4},
    "combination_2": {"d_model": 128, "n_layers": 2, "lr": 5e-4},
}
CIFAR10_BATCH_SIZES = {"combination_1": 8, "combination_2": 8}
CIFAR10_EPOCHS = 20

# Selective Copying Config - FROM PRESENTATION
SELCOPY_CONFIG = {
    "combination_1": {"d_model": 64,  "n_layers": 4, "lr": 1e-3},
    "combination_2": {"d_model": 128, "n_layers": 2, "lr": 5e-4},
}
SELCOPY_BATCH_SIZE = 32
SELCOPY_EPOCHS = 80

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


class GRUBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.gru = nn.GRU(d_model, d_model, 1, batch_first=True, bidirectional=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        normed = self.norm(x)
        output, _ = self.gru(normed)
        return x + self.dropout(output)


class GRUClassifier(L.LightningModule):
    def __init__(self, vocab_size, d_model=128, n_layers=2, num_classes=10, lr=5e-4, weight_decay=0.01, dropout=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.weight_decay = weight_decay
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([GRUBlock(d_model, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )
    
    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)
        return self.classifier(x)
    
    def _shared_step(self, batch, stage):
        logits = self(batch['input_ids'], batch.get('attention_mask'))
        loss = F.cross_entropy(logits, batch['labels'])
        acc = (logits.argmax(dim=-1) == batch['labels']).float().mean()
        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_acc', acc, prog_bar=True)
        return loss
    
    def training_step(self, batch, batch_idx): return self._shared_step(batch, 'train')
    def validation_step(self, batch, batch_idx): return self._shared_step(batch, 'val')
    def test_step(self, batch, batch_idx): return self._shared_step(batch, 'test')
    
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)


class GRUSeqToFixed(nn.Module):
    """For Selective Copying"""
    def __init__(self, vocab_size, output_len, d_model=128, n_layers=2, dropout=0.1):
        super().__init__()
        self.output_len = output_len
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([GRUBlock(d_model, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, input_ids):
        x = self.embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)
        return self.head(x[:, -self.output_len:, :])

In [5]:
class CIFAR10SequentialDataset(Dataset):
    def __init__(self, data, seq_len=1024):
        self.data = data
        self.seq_len = seq_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item[0] if not isinstance(item, dict) else item['input_ids']
        label = item[-1] if not isinstance(item, dict) else item['labels']
        if isinstance(tokens, torch.Tensor): tokens = tokens.tolist()
        if isinstance(label, torch.Tensor): label = label.item()
        
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len
        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [6]:
# Download CIFAR-10
print("Downloading CIFAR-10...")
hf_hub_download(repo_id="monteirot/lra-benchmarks", filename="cifar10_sequential_lra.pt", repo_type="dataset", local_dir=".")
data_cifar10 = torch.load('cifar10_sequential_lra.pt', weights_only=False)
print(f"Vocab: {data_cifar10['vocab_size']} | Classes: {data_cifar10.get('num_classes', 10)}")

cifar10_sequential_lra.pt:   0%|          | 0.00/124M [00:00<?, ?B/s]

Vocab: 257 | Classes: 10


In [7]:
def make_trainer(num_epochs, name):
    return L.Trainer(
        max_epochs=num_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed' if torch.cuda.is_available() else 32,
        logger=CSVLogger(save_dir="logs", name=name),
    )

## 🔬 CIFAR-10 (20 epochs)

In [8]:
results_cifar = {}
num_classes = 10

for combo_name, cfg in CIFAR10_CONFIG.items():
    bs = CIFAR10_BATCH_SIZES[combo_name]
    print(f"\n{'='*60}")
    print(f"  CIFAR-10 — {combo_name}: d={cfg['d_model']}, L={cfg['n_layers']}, lr={cfg['lr']}, bs={bs}")
    print(f"{'='*60}")
    
    train_loader = DataLoader(CIFAR10SequentialDataset(data_cifar10['train'].data), batch_size=bs, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(CIFAR10SequentialDataset(data_cifar10['val'].data), batch_size=bs, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(CIFAR10SequentialDataset(data_cifar10['test'].data), batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    
    model = GRUClassifier(vocab_size=data_cifar10['vocab_size'], d_model=cfg['d_model'], n_layers=cfg['n_layers'], num_classes=num_classes, lr=cfg['lr'])
    print(f"  Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    trainer = make_trainer(CIFAR10_EPOCHS, f"cifar10-{combo_name}")
    trainer.fit(model, train_loader, val_loader)
    trainer.test(model, test_loader)
    
    test_acc = trainer.callback_metrics.get("test_acc")
    if isinstance(test_acc, torch.Tensor): test_acc = test_acc.item()
    
    results_cifar[combo_name] = {"config": cfg, "test_acc": test_acc, "params": sum(p.numel() for p in model.parameters())}
    print(f"\n  ✓ {combo_name}: test_acc = {test_acc:.4f}")

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  CIFAR-10 — combination_1: d=64, L=4, lr=0.0005, bs=8
  Params: 121,418


/home/esen.n/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  100 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 121 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 121 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 25                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

`Trainer.fit` stopped: `max_epochs=20` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.46560001373291016    │
│         test_loss         │    1.7261161804199219     │
└───────────────────────────┴───────────────────────────┘


  ✓ combination_2: test_acc = 0.4656


## 🔬 Selective Copying (80 epochs)

In [9]:
def generate_selective_copying(batch_size, seq_len, num_tokens_to_copy, vocab_size=16, seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    marker_token = vocab_size
    inputs = torch.full((batch_size, seq_len), 0, dtype=torch.long)
    labels = torch.zeros(batch_size, num_tokens_to_copy, dtype=torch.long)
    for b in range(batch_size):
        positions = sorted(random.sample(range(seq_len - 1), num_tokens_to_copy))
        tokens = torch.randint(1, vocab_size, (num_tokens_to_copy,))
        for i, pos in enumerate(positions):
            inputs[b, pos] = tokens[i]
            labels[b, i] = tokens[i]
        inputs[b, -1] = marker_token
    return inputs, labels

NUM_COPY = 6
train_inp, train_lab = generate_selective_copying(512, 128, NUM_COPY, seed=42)
val_inp, val_lab = generate_selective_copying(128, 128, NUM_COPY, seed=99)
test_inp, test_lab = generate_selective_copying(128, 128, NUM_COPY, seed=7)
print(f"Selective Copying: Train={train_inp.shape[0]}, Val={val_inp.shape[0]}, Test={test_inp.shape[0]}")

Selective Copying: Train=512, Val=128, Test=128


In [10]:
def train_selcopy(model, train_inputs, train_labels, epochs, lr, batch_size=32):
    device = next(model.parameters()).device
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    n = train_inputs.shape[0]
    for epoch in tqdm(range(1, epochs + 1), desc="Epochs"):
        model.train()
        perm = torch.randperm(n)
        total_loss, steps = 0, 0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            x, y = train_inputs[idx].to(device), train_labels[idx].to(device)
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            steps += 1
        if epoch % 20 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}  loss = {total_loss/steps:.4f}")

def eval_selcopy(model, inputs, labels):
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        logits = model(inputs.to(device))
        preds = logits.argmax(dim=-1)
        acc = (preds == labels.to(device)).float().mean().item()
    return acc

In [11]:
results_selcopy = {}

for combo_name, cfg in SELCOPY_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"  Selective Copying — {combo_name}: d={cfg['d_model']}, L={cfg['n_layers']}, lr={cfg['lr']}")
    print(f"{'='*60}")
    
    model = GRUSeqToFixed(vocab_size=17, output_len=NUM_COPY, d_model=cfg["d_model"], n_layers=cfg["n_layers"]).to(DEVICE)
    print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")
    
    train_selcopy(model, train_inp, train_lab, SELCOPY_EPOCHS, cfg["lr"], SELCOPY_BATCH_SIZE)
    test_acc = eval_selcopy(model, test_inp, test_lab)
    
    results_selcopy[combo_name] = {"config": cfg, "test_acc": test_acc, "params": sum(p.numel() for p in model.parameters())}
    print(f"\n  ✓ {combo_name}: test_acc = {test_acc:.4f}")


  Selective Copying — combination_1: d=64, L=4, lr=0.001
  Params: 102,336


Epochs:   1%|▏         | 1/80 [00:00<00:09,  8.21it/s]

  Epoch   1  loss = 2.8114


Epochs:  26%|██▋       | 21/80 [00:01<00:05, 10.83it/s]

  Epoch  20  loss = 1.4996


...

  Epoch  40  loss = 1.2167


Epochs:  78%|███████▊  | 62/80 [00:03<00:01, 17.43it/s]

  Epoch  60  loss = 1.1458


Epochs: 100%|██████████| 80/80 [00:04<00:00, 17.42it/s]

  Epoch  80  loss = 1.1097

  ✓ combination_2: test_acc = 0.4362


In [12]:
print("\n" + "="*70)
print("  HPC RESULTS (CIFAR-10 + Selective Copying)")
print("="*70)

mamba_cifar = {"combination_1": 0.46, "combination_2": 0.43}
mamba_selcopy = {"combination_1": 0.3385, "combination_2": 0.3529}

print("\n  📊 CIFAR-10 (20 epochs)")
print(f"  {'Comb':<15} {'Mamba':>10} {'GRU':>10} {'Diff':>10}")
for combo in ["combination_1", "combination_2"]:
    m, g = mamba_cifar[combo], results_cifar[combo]["test_acc"]
    print(f"  {combo:<15} {m:>10.4f} {g:>10.4f} {g-m:>+10.4f}")

print("\n  📊 Selective Copying (80 epochs)")
print(f"  {'Comb':<15} {'Mamba':>10} {'GRU':>10} {'Diff':>10}")
for combo in ["combination_1", "combination_2"]:
    m, g = mamba_selcopy[combo], results_selcopy[combo]["test_acc"]
    print(f"  {combo:<15} {m:>10.4f} {g:>10.4f} {g-m:>+10.4f}")

# Save
torch.save({"cifar10": results_cifar, "selective_copying": results_selcopy}, "gru_hpc_results.pt")
print("\n✓ Saved to gru_hpc_results.pt")


  HPC RESULTS (CIFAR-10 + Selective Copying)

  📊 CIFAR-10 (20 epochs)
  Comb                 Mamba        GRU       Diff
  combination_1       0.4600     0.5088    +0.0488
  combination_2       0.4300     0.4656    +0.0356

  📊 Selective Copying (80 epochs)
  Comb                 Mamba        GRU       Diff
  combination_1       0.3385     0.4271    +0.0886
  combination_2       0.3529     0.4362    +0.0833

✓ Saved to gru_hpc_results.pt


In [13]:
total_time = time.time() - notebook_start_time
print(f"\n⏱️ Total time: {int(total_time//3600)}h {int((total_time%3600)//60)}m {int(total_time%60)}s")


⏱️ Total time: 1h 34m 8s
